# 01 · Fragmentación, replicación y ABM

**Fase 1 — Logística Federal.** Persistencia políglota: PostgreSQL (CP) para lo comercial y
Cassandra (AP) para el tracking. Este notebook demuestra cada mecanismo sobre el clúster real.
Requiere `python tareas.py up` y `python tareas.py seed`.

In [ ]:
import uuid

import pandas as pd
from IPython.display import display

from logistica import abm, tracking
from logistica.conexiones import conectar_pg, sesion_cassandra
from logistica.consultas import consultar
from logistica.indices import explicar

pd.set_option("display.width", 160)

## 1. Topología del clúster

In [ ]:
print("Réplicas conectadas al primario de Central (PostgreSQL):")
display(consultar("centro", "SELECT application_name, state, sync_state FROM pg_stat_replication"))

print("Anillo de Cassandra (un rack por región):")
display(
    pd.DataFrame(
        [
            {"nodo": str(h.address), "rack": h.rack, "datacenter": h.datacenter, "activo": h.is_up}
            for h in sesion_cassandra().cluster.metadata.all_hosts()
        ]
    )
)

## 2. Fragmentación horizontal de `envios` (por región)

`envios` es una tabla **particionada por lista** en `pg-centro`: la partición `centro` es local y
`norte`/`sur` son tablas foráneas (`postgres_fdw`) hacia la tabla física de cada sucursal.

In [ ]:
display(
    consultar(
        "centro",
        "SELECT tableoid::regclass AS fragmento, region, count(*) AS envios "
        "FROM envios GROUP BY 1, 2 ORDER BY 1",
    )
)

**Reglas de corrección**

* **Completitud**: toda fila de la relación global está en algún fragmento.
* **Reconstrucción**: la relación global es la unión de los fragmentos.
* **Disyunción**: ningún envío está en dos fragmentos (cada nodo solo acepta su región).

In [ ]:
global_ = consultar("centro", "SELECT count(*) AS n FROM envios").n[0]
fragmentos = {
    "centro": consultar("centro", "SELECT count(*) AS n FROM envios_centro").n[0],
    "norte": consultar("norte", "SELECT count(*) AS n FROM envios_norte").n[0],
    "sur": consultar("sur", "SELECT count(*) AS n FROM envios_sur").n[0],
}
print("fragmentos:", fragmentos, "| suma:", sum(fragmentos.values()), "| global:", global_)
assert sum(fragmentos.values()) == global_, "falla reconstrucción/completitud"

ajenas = consultar("norte", "SELECT count(*) AS n FROM envios_norte WHERE region <> 'norte'").n[0]
print("filas ajenas en el fragmento Norte:", ajenas)
assert ajenas == 0

try:
    with conectar_pg("norte") as c:
        c.execute(
            "INSERT INTO envios_norte (region, cliente_id, sucursal_origen_id, "
            "sucursal_destino_id, peso_kg, distancia_km) VALUES ('sur', 1, 1, 2, 1, 1)"
        )
except Exception as error:
    print("Disyunción garantizada por el nodo:", type(error).__name__)

**Poda de particiones.** Una consulta con `region = 'norte'` solo toca el nodo Norte; sin el
predicado, el `Append` lee los tres fragmentos (los remotos en paralelo, `async_capable`).

In [ ]:
with conectar_pg("centro") as c:
    print(explicar(c, "SELECT count(*) FROM envios WHERE region = 'norte'"))
    print("-" * 80)
    print(explicar(c, "SELECT count(*) FROM envios"))

## 3. Fragmentación vertical de `clientes`

* `clientes_operativo` (nombre, dirección…) se **replica** a las sucursales.
* `clientes_sensible` (token de tarjeta, scoring) existe **solo** en la sede Central.
* La vista `clientes` reconstruye la relación con un JOIN por la clave primaria.

In [ ]:
def columnas(nodo, tabla):
    filas = consultar(
        nodo,
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_name = %s ORDER BY ordinal_position",
        (tabla,),
    )
    return filas.column_name.tolist() if not filas.empty else []


print("Central · clientes (vista) :", columnas("centro", "clientes"))
print("Norte   · clientes_operativo:", columnas("norte", "clientes_operativo"))
print("Norte   · clientes_sensible :", columnas("norte", "clientes_sensible") or "no existe")

## 4. Replicación

| Qué | Mecanismo | Modo |
|---|---|---|
| Facturación y todo `pg-centro` | streaming físico → `pg-centro-replica` | **síncrona** (`remote_apply`, quórum) |
| Catálogo y `clientes_operativo` | replicación lógica → `pg-norte`, `pg-sur` | **asíncrona**, parcial |
| Tracking | Cassandra RF = 3 | **asíncrona**, eventual |

In [ ]:
display(
    consultar(
        "centro",
        "SELECT name, setting FROM pg_settings WHERE name IN "
        "('synchronous_commit', 'synchronous_standby_names', 'wal_level')",
    )
)
display(consultar("norte", "SELECT subname, subenabled, subfailover FROM pg_subscription"))

## 5. ABM relacional

In [ ]:
email = f"demo-{uuid.uuid4().hex[:8]}@ejemplo.com.ar"
cliente = abm.alta_cliente(
    nombre="Cliente Demo",
    email=email,
    direccion="San Martín 100",
    ciudad="Salta",
    provincia="Salta",
    region="norte",
    token_tarjeta="tok_demo",
    ultimos4="4242",
    limite_credito=150000,
    scoring=720,
)
print("ALTA   cliente", cliente)
display(pd.DataFrame([abm.obtener_cliente(cliente)]))

abm.modificar_cliente(cliente, direccion="Belgrano 200", scoring=760)
print("MODIF. cliente", cliente)
display(pd.DataFrame([abm.obtener_cliente(cliente)]))

In [ ]:
import time

# La sucursal conoce al cliente por replicación asíncrona (sin datos sensibles).
for _ in range(50):
    en_norte = abm.obtener_cliente(cliente, "norte")
    if en_norte:
        break
    time.sleep(0.2)
print("Visible en pg-norte:", en_norte is not None, "· columnas sensibles:", "scoring" in en_norte)

### Envío + factura con **two-phase commit**

El envío vive en el nodo de su región y la factura en Central: son dos nodos, así que la
atomicidad se logra con `PREPARE TRANSACTION` / `COMMIT PREPARED`.

In [ ]:
envio, factura = abm.alta_envio_facturado(
    region="norte",
    cliente_id=cliente,
    sucursal_origen_id=1,
    sucursal_destino_id=3,
    peso_kg=12.5,
    distancia_km=620,
    monto=4300,
)
print("ALTA envío", envio, "factura", factura)
display(pd.DataFrame([abm.obtener_envio("norte", envio)]))
print(
    "Transacciones en duda:",
    abm.transacciones_en_duda("norte"),
    abm.transacciones_en_duda("centro"),
)

In [ ]:
# Falla simulada: el monto negativo viola un CHECK en Central después de haber
# insertado el envío en Norte. El protocolo debe deshacer ambos.
antes = consultar(
    "norte", "SELECT count(*) AS n FROM envios_norte WHERE cliente_id = %s", (cliente,)
).n[0]
try:
    abm.alta_envio_facturado(
        region="norte",
        cliente_id=cliente,
        sucursal_origen_id=1,
        sucursal_destino_id=3,
        peso_kg=1,
        distancia_km=1,
        monto=-1,
    )
except Exception as error:
    print("Operación abortada:", type(error).__name__)
despues = consultar(
    "norte", "SELECT count(*) AS n FROM envios_norte WHERE cliente_id = %s", (cliente,)
).n[0]
print(f"envíos del cliente antes={antes} después={despues}")
assert antes == despues

In [ ]:
abm.modificar_envio("norte", envio, estado="en_transito", vehiculo_id=3)
abm.registrar_pago(factura, 4300, "transferencia")
print(
    "MODIF. envío ->",
    abm.obtener_envio("norte", envio)["estado"],
    "· factura ->",
    abm.obtener_factura_de_envio(envio)["estado"],
)

abm.baja_envio("norte", envio)
print(
    "BAJA   envío ->",
    abm.obtener_envio("norte", envio),
    "· factura ->",
    abm.obtener_factura_de_envio(envio)["estado"],
)

abm.baja_cliente(cliente)
print("BAJA   cliente (lógica) -> activo =", abm.obtener_cliente(cliente)["activo"])

## 6. ABM del subsistema NoSQL (Cassandra)

Diseño guiado por consultas: `eventos_por_paquete` se particiona por `id_paquete`
(sharding por hash del token); dentro de cada partición los eventos quedan ordenados por tiempo.

In [ ]:
display(
    pd.DataFrame(
        sesion_cassandra().execute(
            "SELECT table_name, default_time_to_live FROM system_schema.tables WHERE keyspace_name = 'tracking'"
        )
    )
)

paquete = uuid.uuid4()
t1 = tracking.registrar_evento(paquete, "ingreso_deposito", "sur", -38.95, -68.06, sucursal_id=11)
tracking.registrar_evento(paquete, "en_transito", "sur", -38.8, -67.9)
print("ALTA/consulta:")
display(pd.DataFrame(tracking.historial_paquete(paquete))[["ts", "tipo", "region", "lat", "lon"]])

tracking.registrar_evento(paquete, "incidencia", "sur", -38.95, -68.06, ts=t1, detalle="corregido")
print("MODIF. (upsert sobre la misma clave):", tracking.historial_paquete(paquete)[-1]["tipo"])

tracking.eliminar_evento(paquete, t1)
print("BAJA:", len(tracking.historial_paquete(paquete)), "evento restante")

token = sesion_cassandra().execute(
    "SELECT token(id_paquete) AS hash_token FROM tracking.eventos_por_paquete LIMIT 3"
)
print(
    "Tokens (determinan qué nodos guardan cada partición):", [fila["hash_token"] for fila in token]
)